**Analysis of the pypsa .nc files**

Here we explore and plot electricity prices in different scenarios

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pypsa
import matplotlib.pyplot as plt

# -------------------------------------------------------------------
# 0) SETTINGS
# -------------------------------------------------------------------
warnings.filterwarnings("ignore", category=FutureWarning, module="pypsa")

# -------------------------------------------------------------------
# 0) INPUT LIST
# -------------------------------------------------------------------
### From "Flattening the peak demand curve through energy efficient buildings - A holistic approach towards net-zero carbon"
# retro_tes - 2030
# path_nc = r'data/carriers_data/electricity/retro-tes_2030.nc'
# retro_tes - 2040
# path_nc = r'data/carriers_data/electricity/retro-tes_2040.nc'
# retro_tes - 2050
# path_nc = r'data/carriers_data/electricity/retro-tes_2050.nc'
# flexible - 2030
# path_nc = r'data/carriers_data/electricity/flexible_2030.nc'
# flexible - 2040
path_nc = r'data/carriers_data/electricity/flexible_2040.nc'
# flexible - 2050
# path_nc = r'data/carriers_data/electricity/flexible_2050.nc'
# flexible_moderate - 2030
# path_nc = r'data/carriers_data/electricity/flexible-moderate_2030.nc'
# flexible_moderate - 2040
# path_nc = r'data/carriers_data/electricity/flexible-moderate_2040.nc'
# flexible_moderate - 2050
# path_nc = r'data/carriers_data/electricity/flexible-moderate_2050.nc'
# rigid - 2030
# path_nc = r'data/carriers_data/electricity/rigid_2030.nc'
# rigid - 2040
# path_nc = r'data/carriers_data/electricity/rigid_2040.nc'
# rigid - 2050
# path_nc = r'data/carriers_data/electricity/rigid_2050.nc'

### From "Strategic deployment of solar photovoltaics for achieving self-sufficiency in Europe throughout the energy transition"
# base - 2030
# path_nc = r"data\carriers_data\electricity\elec_s370_37m_lv1.1__2H-T-H-B-I-A-solar+p2-dist1-highrooftop3-cb29ex0_2030.nc"
# base - 2040
# path_nc = r"data\carriers_data\electricity\elec_s370_37m_lv1.1__2H-T-H-B-I-A-solar+p2-dist1-highrooftop3-cb29ex0_2040.nc"
# base - 2050
# path_nc = r"data\carriers_data\electricity\elec_s370_37m_lv1.1__2H-T-H-B-I-A-solar+p2-dist1-highrooftop3-cb29ex0_2050.nc"
# newsolar - 2030
# path_nc = r"data\carriers_data\electricity\elec_s370_37m_lv1.1__2H-T-H-B-I-A-solar+p2-dist1-highrooftop3-newsolar-cb29ex0_2030.nc"
# newsolar - 2040
# path_nc = r"data\carriers_data\electricity\elec_s370_37m_lv1.1__2H-T-H-B-I-A-solar+p2-dist1-highrooftop3-newsolar-cb29ex0_2040.nc"
# newsolar - 2050
# path_nc = r"data\carriers_data\electricity\elec_s370_37m_lv1.1__2H-T-H-B-I-A-solar+p2-dist1-highrooftop3-newsolar-cb29ex0_2050.nc"


COUNTRIES = ["NL", "BE", "DE"]

USE_WEIGHTED = True
WEIGHTED_NAN_THRESHOLD = 0.05   # fallback to mean if >5% NaN hours

PRICE_THRESHOLDS = [300, 500]

# Filters for spike diagnostics: remove numerical dust / irrelevant tiny assets
MIN_LINE_FLOW_MW = 10.0
MIN_LINE_CAP_MW = 100.0

MIN_DC_FLOW_MW = 10.0
MIN_DC_CAP_MW = 100.0

MIN_LOCAL_FLOW_MW = 10.0
MIN_LOCAL_CAP_MW = 10.0

CONGESTION_THRESHOLD = 0.98

TOP_N_LINES = 5
TOP_N_DC = 5
TOP_N_LOCAL = 8

SHOW_PLOT = True
DIAGNOSE_COUNTRIES = ["BE", "NL", "DE"]


# -------------------------------------------------------------------
# 1) HELPERS
# -------------------------------------------------------------------
def get_snapshot_hours(n):
    """
    Returns a pd.Series indexed by n.snapshots with the number of hours
    represented by each snapshot.
    """
    snapshots = n.snapshots

    if isinstance(snapshots, pd.DatetimeIndex) and len(snapshots) >= 2:
        diffs = snapshots.to_series().diff().dt.total_seconds().div(3600.0)
        if diffs.iloc[1:].notna().all():
            diffs.iloc[0] = diffs.iloc[1]
            return diffs.rename("hours")

    if hasattr(n, "snapshot_weightings") and n.snapshot_weightings is not None:
        for col in ["objective", "generators", "stores"]:
            if col in n.snapshot_weightings.columns:
                return n.snapshot_weightings[col].astype(float).rename("hours")

    return pd.Series(1.0, index=snapshots, name="hours")


def weighted_time_mean(series, weights):
    mask = series.notna()
    if mask.sum() == 0:
        return np.nan
    return (series[mask] * weights[mask]).sum() / weights[mask].sum()


def fmt_hours(x):
    return f"{x:.1f} h"


def fmt_float(x, nd=2):
    if pd.isna(x):
        return "NaN"
    return f"{x:.{nd}f}"


def print_header(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def safe_divide(num, den):
    out = num / den
    return out.replace([np.inf, -np.inf], np.nan)


def top_df(df, n=5, cols=None):
    if df is None or df.empty:
        return None
    if cols is not None:
        cols = [c for c in cols if c in df.columns]
        return df.loc[:, cols].head(n)
    return df.head(n)


# -------------------------------------------------------------------
# 2) LOAD NETWORK
# -------------------------------------------------------------------
n = pypsa.Network(path_nc)
snapshot_hours = get_snapshot_hours(n)
dt_unique = np.sort(snapshot_hours.unique())

print_header("NETWORK SUMMARY")
print(f"File: {Path(path_nc).name}")
print(f"Snapshots: {len(n.snapshots)}")
print(f"Timestep(s): {', '.join([str(x) for x in dt_unique])} h")
print(f"Represented hours: {snapshot_hours.sum():.1f} h")
print(f"Buses: {len(n.buses)} | Loads: {len(n.loads)} | Lines: {len(n.lines)} | Links: {len(n.links)}")


# -------------------------------------------------------------------
# 3) FILTER ELECTRIC AC BUSES
# -------------------------------------------------------------------
ac_buses = n.buses.index[(n.buses["carrier"] == "AC") & (n.buses["country"] != "")]
ac_buses_c = ac_buses[n.buses.loc[ac_buses, "country"].isin(COUNTRIES)]
bus_country = n.buses.loc[ac_buses_c, "country"]

print_header("AC BUS SELECTION")
print(f"Total AC country buses: {len(ac_buses)}")
print(f"Selected AC buses ({COUNTRIES}): {len(ac_buses_c)}")
print(n.buses.loc[ac_buses_c, "country"].value_counts().rename("n_buses"))


# -------------------------------------------------------------------
# 4) NODAL PRICES AND NATIONAL SERIES
# -------------------------------------------------------------------
p_ac = n.buses_t.marginal_price[ac_buses_c].copy()

# Path 1: simple mean across AC buses of the country
p_country_mean = p_ac.T.groupby(bus_country).mean().T

# Path 2: median across AC buses of the country
p_country_median = p_ac.T.groupby(bus_country).median().T

# Path 3: load-weighted average across AC buses of the country
# WARNING: if electric loads are on LV buses, denominator can be zero
load_bus = n.loads_t.p_set.T.groupby(n.loads.bus).sum().T
load_bus_ac = load_bus.reindex(columns=ac_buses_c).fillna(0.0)

num = (p_ac * load_bus_ac).T.groupby(bus_country).sum().T
den = load_bus_ac.T.groupby(bus_country).sum().T
p_country_weighted = safe_divide(num, den)

p_country = p_country_weighted.copy() if USE_WEIGHTED else p_country_mean.copy()

# keep track of which aggregation is actually used
aggregation_used = {}

for c in COUNTRIES:
    if c not in p_country.columns:
        continue

    if USE_WEIGHTED:
        nan_hours = snapshot_hours[p_country[c].isna()].sum()
        nan_share = nan_hours / snapshot_hours.sum()

        if nan_share > WEIGHTED_NAN_THRESHOLD:
            p_country[c] = p_country_mean[c]
            aggregation_used[c] = "mean (weighted fallback)"
        else:
            aggregation_used[c] = "weighted"
    else:
        aggregation_used[c] = "mean"


# -------------------------------------------------------------------
# 5) COUNTRY SUMMARY TABLE
# -------------------------------------------------------------------
summary_rows = []

for c in COUNTRIES:
    if c not in p_country.columns:
        continue

    s = p_country[c]
    row = {
        "country": c,
        "aggregation": aggregation_used.get(c, "n/a"),
        "avg_price": weighted_time_mean(s, snapshot_hours),
        "p95": s.quantile(0.95),
        "p99": s.quantile(0.99),
        "max": s.max(),
        "zero_load_hours": snapshot_hours[den[c] == 0].sum() if c in den.columns else np.nan,
        "nan_hours": snapshot_hours[s.isna()].sum(),
    }

    for thr in PRICE_THRESHOLDS:
        row[f"hours_gt_{thr}"] = snapshot_hours[s > thr].sum()

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index("country")

print_header("PRICE SUMMARY")
display_cols = [
    "aggregation", "avg_price", "p95", "p99", "max",
    "zero_load_hours", "nan_hours"
] + [f"hours_gt_{thr}" for thr in PRICE_THRESHOLDS]

summary_to_print = summary_df[display_cols].copy()
num_cols = summary_to_print.select_dtypes(include=[np.number]).columns
summary_to_print[num_cols] = summary_to_print[num_cols].round(2)
print(summary_to_print)


# -------------------------------------------------------------------
# 6) PLOT
# -------------------------------------------------------------------
if SHOW_PLOT:
    plt.figure(figsize=(12, 6))

    labels = {"NL": "Netherlands", "BE": "Belgium", "DE": "Germany"}
    for c in COUNTRIES:
        if c in p_country.columns:
            plt.plot(p_country.index, p_country[c].values, label=labels.get(c, c))

    plt.xlabel("Time")
    plt.ylabel("Electricity price (€/MWh)")
    plt.title("National electricity prices (aggregated from AC nodal prices)")
    plt.legend()
    plt.tight_layout()
    plt.show()


# -------------------------------------------------------------------
# 7) SPIKE DIAGNOSTICS
# -------------------------------------------------------------------
def get_country_main_bus(country_code, t_spike):
    buses_c = ac_buses_c[n.buses.loc[ac_buses_c, "country"] == country_code]
    if len(buses_c) == 0:
        return None, None

    p_nodes = n.buses_t.marginal_price[buses_c].loc[t_spike].sort_values(ascending=False)
    if p_nodes.empty:
        return None, None

    main_bus = p_nodes.index[0]
    return main_bus, p_nodes


def build_line_loading_table(main_bus, t_spike):
    lines = n.lines.index[(n.lines.bus0 == main_bus) | (n.lines.bus1 == main_bus)]
    if len(lines) == 0:
        return pd.DataFrame()

    flow = n.lines_t.p0.loc[t_spike, lines].abs()
    s_nom_eff = n.lines.loc[lines, "s_nom_opt"].fillna(n.lines.loc[lines, "s_nom"])
    s_max_pu = n.lines.loc[lines, "s_max_pu"].fillna(1.0)
    cap_eff = s_nom_eff * s_max_pu
    loading_eff = safe_divide(flow, cap_eff)

    out = pd.DataFrame({
        "flow_MW": flow,
        "cap_eff_MW": cap_eff,
        "loading": loading_eff,
        "bus0": n.lines.loc[lines, "bus0"],
        "bus1": n.lines.loc[lines, "bus1"],
    }).dropna(subset=["loading"])

    out = out[(out["flow_MW"] >= MIN_LINE_FLOW_MW) & (out["cap_eff_MW"] >= MIN_LINE_CAP_MW)]
    out = out.sort_values("loading", ascending=False)
    return out


def build_link_loading_table(main_bus, t_spike):
    links = n.links.index[(n.links.bus0 == main_bus) | (n.links.bus1 == main_bus)]
    if len(links) == 0:
        return pd.DataFrame()

    p_nom_eff = n.links.loc[links, "p_nom_opt"].fillna(n.links.loc[links, "p_nom"])
    flow = n.links_t.p0.loc[t_spike, links].abs()
    loading = safe_divide(flow, p_nom_eff)

    out = pd.DataFrame({
        "flow_MW": flow,
        "cap_eff_MW": p_nom_eff,
        "loading": loading,
        "carrier": n.links.loc[links, "carrier"].astype(str),
        "bus0": n.links.loc[links, "bus0"],
        "bus1": n.links.loc[links, "bus1"],
        "efficiency": n.links.loc[links, "efficiency"] if "efficiency" in n.links.columns else np.nan,
        "marginal_cost": n.links.loc[links, "marginal_cost"] if "marginal_cost" in n.links.columns else np.nan,
    }).dropna(subset=["loading"])

    return out.sort_values("loading", ascending=False)


def split_links_tables(out_links):
    if out_links.empty:
        return pd.DataFrame(), pd.DataFrame()

    # DC interconnectors
    dc_mask = out_links["carrier"].str.upper().eq("DC")
    dc = out_links[dc_mask].copy()
    dc = dc[(dc["flow_MW"] >= MIN_DC_FLOW_MW) & (dc["cap_eff_MW"] >= MIN_DC_CAP_MW)]
    dc = dc.sort_values("loading", ascending=False)

    # local supply/conversion links
    local = out_links[~dc_mask].copy()
    local = local[(local["flow_MW"] >= MIN_LOCAL_FLOW_MW) & (local["cap_eff_MW"] >= MIN_LOCAL_CAP_MW)]
    local = local.sort_values("loading", ascending=False)

    return dc, local


def diagnose_spike(country_code):
    if country_code not in p_country.columns:
        print(f"\n{country_code}: not available in price table.")
        return

    s = p_country[country_code]
    t_spike = s.idxmax()
    spike_val = s.loc[t_spike]

    main_bus, p_nodes = get_country_main_bus(country_code, t_spike)
    if main_bus is None:
        print(f"\n{country_code}: no AC bus found.")
        return

    lines_tbl = build_line_loading_table(main_bus, t_spike)
    links_tbl = build_link_loading_table(main_bus, t_spike)
    dc_tbl, local_tbl = split_links_tables(links_tbl)

    print_header(f"SPIKE REPORT - {country_code}")
    print(f"Peak timestamp : {t_spike}")
    print(f"Peak price     : {spike_val:.2f} €/MWh")
    print(f"Main AC bus    : {main_bus}")
    print(f"Aggregation    : {aggregation_used.get(country_code, 'n/a')}")
    print()

    if p_nodes is not None:
        print("Nodal prices on AC buses in country at spike:")
        print(p_nodes.round(2).to_string())
        print()

    if not lines_tbl.empty:
        n_cong_lines = (lines_tbl["loading"] >= CONGESTION_THRESHOLD).sum()
        print(f"Top AC lines near bus ({n_cong_lines} with loading >= {CONGESTION_THRESHOLD:.2f}):")
        print(
            top_df(
                lines_tbl.round(3),
                n=TOP_N_LINES,
                cols=["flow_MW", "cap_eff_MW", "loading", "bus0", "bus1"]
            ).to_string()
        )
        print()
    else:
        print("Top AC lines near bus: none above significance thresholds.\n")

    if not dc_tbl.empty:
        n_cong_dc = (dc_tbl["loading"] >= CONGESTION_THRESHOLD).sum()
        print(f"Top DC interconnectors ({n_cong_dc} with loading >= {CONGESTION_THRESHOLD:.2f}):")
        print(
            top_df(
                dc_tbl.round(3),
                n=TOP_N_DC,
                cols=["flow_MW", "cap_eff_MW", "loading", "bus0", "bus1"]
            ).to_string()
        )
        print()
    else:
        print("Top DC interconnectors: none above significance thresholds.\n")

    if not local_tbl.empty:
        n_sat_local = (local_tbl["loading"] >= CONGESTION_THRESHOLD).sum()
        print(f"Top local supply/conversion links ({n_sat_local} with loading >= {CONGESTION_THRESHOLD:.2f}):")
        print(
            top_df(
                local_tbl.round(3),
                n=TOP_N_LOCAL,
                cols=["carrier", "flow_MW", "cap_eff_MW", "loading", "marginal_cost", "bus0", "bus1"]
            ).to_string()
        )
        print()
    else:
        print("Top local supply/conversion links: none above significance thresholds.\n")


# -------------------------------------------------------------------
# 8) RUN SPIKE REPORTS
# -------------------------------------------------------------------
for c in DIAGNOSE_COUNTRIES:
    diagnose_spike(c)

This script builds **all-in electricity price time series** for **NL, BE, and DE** by combining hourly **PyPSA AC marginal prices** with fixed **country-specific TSO transmission tariffs** taken from the European transmission tariff report (see Obsidian page on this). For each scenario network (`.nc`), it filters AC buses, aggregates nodal marginal prices by country, and adds the corresponding tariff (expressed in **€/MWh**) to obtain an adjusted electricity price series. The resulting hourly prices are exported to CSV, together with a summary file containing basic statistics such as mean price, maximum price, and the number of hours above selected thresholds.

In [ ]:
import os
import re
import warnings
 
import numpy as np
import pandas as pd
import pypsa
 
warnings.filterwarnings("ignore", category=FutureWarning, module="pypsa")
 
# -------------------------------------------------------------------
# SETTINGS
# -------------------------------------------------------------------
COUNTRIES = ["NL", "BE", "DE"]
 
USE_WEIGHTED = True
WEIGHTED_NAN_THRESHOLD = 0.05   # fallback to mean if >5% NaN hours
 
TSO_TARIFFS = {
    "BE": 2.62,   # €/MWh
    "DE": 5.35,   # €/MWh
    "NL": 0.00,   # €/MWh
}

# From "Flattening the peak demand curve through energy efficient buildings: A holistic approach towards net-zero carbon"
# nc_files = [
#     r"data/carriers_data/electricity/flexible_2030.nc",
#     r"data/carriers_data/electricity/flexible_2040.nc",
#     r"data/carriers_data/electricity/flexible_2050.nc",
#     r"data/carriers_data/electricity/flexible-moderate_2030.nc",
#     r"data/carriers_data/electricity/flexible-moderate_2040.nc",
#     r"data/carriers_data/electricity/flexible-moderate_2050.nc",
#     r"data/carriers_data/electricity/retro-tes_2030.nc",
#     r"data/carriers_data/electricity/retro-tes_2040.nc",
#     r"data/carriers_data/electricity/retro-tes_2050.nc",
#     r"data/carriers_data/electricity/rigid_2030.nc",
#     r"data/carriers_data/electricity/rigid_2040.nc",
#     r"data/carriers_data/electricity/rigid_2050.nc",
# ]
### From "Strategic deployment of solar photovoltaics for achieving self-sufficiency in Europe throughout the energy transition"
nc_files = [
    r"data/carriers_data/electricity/elec_s370_37m_lv1.1__2H-T-H-B-I-A-solar+p2-dist1-highrooftop3-cb29ex0_2030.nc",
    r"data/carriers_data/electricity/elec_s370_37m_lv1.1__2H-T-H-B-I-A-solar+p2-dist1-highrooftop3-cb29ex0_2040.nc",
    r"data/carriers_data/electricity/elec_s370_37m_lv1.1__2H-T-H-B-I-A-solar+p2-dist1-highrooftop3-cb29ex0_2050.nc",
    r"data/carriers_data/electricity/elec_s370_37m_lv1.1__2H-T-H-B-I-A-solar+p2-dist1-highrooftop3-newsolar-cb29ex0_2030.nc",
    r"data/carriers_data/electricity/elec_s370_37m_lv1.1__2H-T-H-B-I-A-solar+p2-dist1-highrooftop3-newsolar-cb29ex0_2040.nc",
    r"data/carriers_data/electricity/elec_s370_37m_lv1.1__2H-T-H-B-I-A-solar+p2-dist1-highrooftop3-newsolar-cb29ex0_2050.nc",
]
 
out_dir = r"data/carriers_data/electricity"
 
# -------------------------------------------------------------------
# HELPERS  (identical to analysis script)
# -------------------------------------------------------------------
def get_snapshot_hours(n: pypsa.Network) -> pd.Series:
    """Hours represented by each snapshot."""
    snapshots = n.snapshots
    if isinstance(snapshots, pd.DatetimeIndex) and len(snapshots) >= 2:
        diffs = snapshots.to_series().diff().dt.total_seconds().div(3600.0)
        if diffs.iloc[1:].notna().all():
            diffs.iloc[0] = diffs.iloc[1]
            return diffs.rename("hours")
    if hasattr(n, "snapshot_weightings") and n.snapshot_weightings is not None:
        for col in ["objective", "generators", "stores"]:
            if col in n.snapshot_weightings.columns:
                return n.snapshot_weightings[col].astype(float).rename("hours")
    return pd.Series(1.0, index=snapshots, name="hours")
 
 
def safe_divide(num: pd.DataFrame, den: pd.DataFrame) -> pd.DataFrame:
    out = num / den
    return out.replace([np.inf, -np.inf], np.nan)
 
 
# -------------------------------------------------------------------
# CORE FUNCTIONS
# -------------------------------------------------------------------
def extract_ac_country_prices(
    n: pypsa.Network,
    countries: list = COUNTRIES,
    use_weighted: bool = USE_WEIGHTED,
    nan_threshold: float = WEIGHTED_NAN_THRESHOLD,
) -> tuple[pd.DataFrame, dict]:
    """
    Extract electric prices (carrier='AC') and aggregate by country.
 
    Aggregation logic (same as analysis script):
      - primary:  load-weighted mean across AC buses of the country
      - fallback: simple mean if weighted produces >nan_threshold NaN hours
 
    Returns
    -------
    p_country : DataFrame  (datetime index, country columns) in €/MWh
    agg_used  : dict       country -> aggregation method actually used
    """
    # 1) Filter AC buses
    ac_buses = n.buses.index[
        (n.buses["carrier"] == "AC") & (n.buses["country"] != "")
    ]
    ac_buses = ac_buses[n.buses.loc[ac_buses, "country"].isin(countries)]
 
    if len(ac_buses) == 0:
        raise ValueError(
            "No AC buses found for the requested countries. "
            "Check carrier/country fields in the network."
        )
 
    bus_country = n.buses.loc[ac_buses, "country"]
    snapshot_hours = get_snapshot_hours(n)
    total_hours = snapshot_hours.sum()
 
    # 2) Nodal prices
    p_ac = n.buses_t.marginal_price[ac_buses].copy()
 
    # 3) Simple mean (always computed as fallback)
    p_mean = p_ac.T.groupby(bus_country).mean().T
 
    # 4) Load-weighted mean
    load_bus = n.loads_t.p_set.T.groupby(n.loads.bus).sum().T
    load_bus_ac = load_bus.reindex(columns=ac_buses).fillna(0.0)
    num = (p_ac * load_bus_ac).T.groupby(bus_country).sum().T
    den = load_bus_ac.T.groupby(bus_country).sum().T
    p_weighted = safe_divide(num, den)
 
    # 5) Choose aggregation per country
    p_country = p_weighted.copy() if use_weighted else p_mean.copy()
    agg_used = {}
 
    for c in countries:
        if c not in p_country.columns:
            continue
        if use_weighted:
            nan_hours = snapshot_hours[p_country[c].isna()].sum()
            if nan_hours / total_hours > nan_threshold:
                p_country[c] = p_mean[c]
                agg_used[c] = "mean (weighted fallback)"
            else:
                agg_used[c] = "weighted"
        else:
            agg_used[c] = "mean"
 
    # 6) Reorder columns and ensure datetime index
    p_country = p_country.reindex(columns=countries)
    p_country.index = pd.to_datetime(p_country.index)
 
    return p_country, agg_used
 
 
def scenario_name_from_path(path_nc: str) -> str:
    """flexible_2050.nc  ->  flexible_2050"""
    return re.sub(r"\.nc$", "", os.path.basename(path_nc))
 
 
def export_scenario_csv(
    path_nc: str,
    out_dir: str,
    tso_tariffs: pd.Series,
    countries: list = COUNTRIES,
) -> tuple[pd.DataFrame, dict]:
    """
    Load network, extract prices with consistent aggregation, add fixed TSO
    tariffs, and export to CSV.
 
    Returns
    -------
    df_allin : DataFrame with all-in prices (market + TSO) in €/MWh
    agg_used : dict of aggregation method per country
    """
    n = pypsa.Network(path_nc)
 
    df_market, agg_used = extract_ac_country_prices(n, countries=countries)
 
    # Add fixed country-specific TSO tariffs (€/MWh, constant across all hours)
    df_allin = df_market.add(tso_tariffs, axis="columns")
 
    scen = scenario_name_from_path(path_nc)
    out_csv = os.path.join(out_dir, f"prices_AC_allin_{scen}.csv")
    df_allin.to_csv(out_csv, index=True)
 
    agg_str = " | ".join(f"{c}: {v}" for c, v in agg_used.items())
    print(f"[OK] {scen:<45}  shape={df_allin.shape}  agg=({agg_str})")
 
    return df_allin, agg_used
 
 
def summarize_prices(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "mean":         df.mean(),
        "max":          df.max(),
        "hours_gt_300": (df > 300).sum(),
        "hours_gt_500": (df > 500).sum(),
    })
 
 
# -------------------------------------------------------------------
# MAIN LOOP
# -------------------------------------------------------------------
tso_tariffs = pd.Series(TSO_TARIFFS).reindex(COUNTRIES)
all_summaries = []
 
for path_nc in nc_files:
    if not os.path.exists(path_nc):
        print(f"[SKIP] missing file: {path_nc}")
        continue
 
    df_allin, agg_used = export_scenario_csv(
        path_nc,
        out_dir,
        tso_tariffs,
        countries=COUNTRIES,
    )
 
    summ = summarize_prices(df_allin)
    summ["scenario"] = scenario_name_from_path(path_nc)
    all_summaries.append(summ.reset_index().rename(columns={"index": "country"}))
 
if all_summaries:
    summary_df = pd.concat(all_summaries, ignore_index=True)
    summary_path = os.path.join(out_dir, "summary_prices_allin.csv")
    summary_df.to_csv(summary_path, index=False)
    print(f"\n[OK] Saved summary: {summary_path}")

In [ ]:
import os
import re
import warnings

import numpy as np
import pandas as pd
import pypsa

warnings.filterwarnings("ignore", category=FutureWarning, module="pypsa")

# -------------------------------------------------------------------
# SETTINGS
# -------------------------------------------------------------------
COUNTRIES = ["NL", "BE", "DE"]

CI_THRESHOLDS = [50, 100]   # kgCO2/MWh — used in summary (analogous to PRICE_THRESHOLDS)

# From "Flattening the peak demand curve through energy efficient buildings: A holistic approach towards net-zero carbon"
# nc_files = [
#     r"data/carriers_data/electricity/flexible_2030.nc",
#     r"data/carriers_data/electricity/flexible_2040.nc",
#     r"data/carriers_data/electricity/flexible_2050.nc",
#     r"data/carriers_data/electricity/flexible-moderate_2030.nc",
#     r"data/carriers_data/electricity/flexible-moderate_2040.nc",
#     r"data/carriers_data/electricity/flexible-moderate_2050.nc",
#     r"data/carriers_data/electricity/retro-tes_2030.nc",
#     r"data/carriers_data/electricity/retro-tes_2040.nc",
#     r"data/carriers_data/electricity/retro-tes_2050.nc",
#     r"data/carriers_data/electricity/rigid_2030.nc",
#     r"data/carriers_data/electricity/rigid_2040.nc",
#     r"data/carriers_data/electricity/rigid_2050.nc",
# ]

# From "Strategic deployment of solar photovoltaics for achieving self-sufficiency in Europe throughout the energy transition"
nc_files = [
    r"data/carriers_data/electricity/elec_s370_37m_lv1.1__2H-T-H-B-I-A-solar+p2-dist1-highrooftop3-cb29ex0_2030.nc",
    r"data/carriers_data/electricity/elec_s370_37m_lv1.1__2H-T-H-B-I-A-solar+p2-dist1-highrooftop3-cb29ex0_2040.nc",
    r"data/carriers_data/electricity/elec_s370_37m_lv1.1__2H-T-H-B-I-A-solar+p2-dist1-highrooftop3-cb29ex0_2050.nc",
    r"data/carriers_data/electricity/elec_s370_37m_lv1.1__2H-T-H-B-I-A-solar+p2-dist1-highrooftop3-newsolar-cb29ex0_2030.nc",
    r"data/carriers_data/electricity/elec_s370_37m_lv1.1__2H-T-H-B-I-A-solar+p2-dist1-highrooftop3-newsolar-cb29ex0_2040.nc",
    r"data/carriers_data/electricity/elec_s370_37m_lv1.1__2H-T-H-B-I-A-solar+p2-dist1-highrooftop3-newsolar-cb29ex0_2050.nc",
]

out_dir = r"data/carriers_data/electricity"


# -------------------------------------------------------------------
# HELPERS  (identical to export_prices.py)
# -------------------------------------------------------------------
def get_snapshot_hours(n: pypsa.Network) -> pd.Series:
    """Hours represented by each snapshot."""
    snapshots = n.snapshots
    if isinstance(snapshots, pd.DatetimeIndex) and len(snapshots) >= 2:
        diffs = snapshots.to_series().diff().dt.total_seconds().div(3600.0)
        if diffs.iloc[1:].notna().all():
            diffs.iloc[0] = diffs.iloc[1]
            return diffs.rename("hours")
    if hasattr(n, "snapshot_weightings") and n.snapshot_weightings is not None:
        for col in ["objective", "generators", "stores"]:
            if col in n.snapshot_weightings.columns:
                return n.snapshot_weightings[col].astype(float).rename("hours")
    return pd.Series(1.0, index=snapshots, name="hours")


def safe_divide(num: pd.DataFrame, den: pd.DataFrame) -> pd.DataFrame:
    out = num / den
    return out.replace([np.inf, -np.inf], np.nan)


def scenario_name_from_path(path_nc: str) -> str:
    """flexible_2050.nc  ->  flexible_2050"""
    return re.sub(r"\.nc$", "", os.path.basename(path_nc))


# -------------------------------------------------------------------
# HELPERS  (CI-specific)
# -------------------------------------------------------------------
def _get_ts(component_t, key):
    return getattr(component_t, key) if hasattr(component_t, key) else component_t[key]


def _fix_sign(df: pd.DataFrame) -> pd.DataFrame:
    """If series is mostly negative, flip sign (sign-convention safeguard)."""
    if df.size == 0:
        return df
    return (-df) if df.to_numpy().mean() < 0 else df


# -------------------------------------------------------------------
# CORE FUNCTIONS
# -------------------------------------------------------------------
def extract_ac_country_carbon_intensity(
    n: pypsa.Network,
    countries: list = COUNTRIES,
    mode: str = "net",   # "net" or "gross"
) -> dict:
    """
    Production-based carbon intensity on AC buses by country.

    mode:
      - "net"  : CO2 -> 'co2 atmosphere' only
      - "gross": CO2 -> atmosphere + stored + sequestered

    Returns dict with:
      ci          (kgCO2/MWh)  — hourly carbon intensity
      E_MWh       (MWh/h)      — electricity produced on AC buses
      CO2_atm_t   (tCO2/h)
      CO2_store_t (tCO2/h)
      CO2_seq_t   (tCO2/h)
    """
    # 1) Filter AC buses
    ac_buses = n.buses.index[
        (n.buses["carrier"] == "AC") & (n.buses["country"] != "")
    ]
    ac_buses_c = ac_buses[n.buses.loc[ac_buses, "country"].isin(countries)]
    if len(ac_buses_c) == 0:
        raise ValueError(
            "No AC buses found for the requested countries. "
            "Check carrier/country fields in the network."
        )

    bus_country = n.buses.loc[ac_buses_c, "country"]

    # -------------------------
    # (A) Electricity denominator (MWh/h)
    # -------------------------
    # Generators on AC buses
    gens = n.generators.index[n.generators.bus.isin(ac_buses_c)]
    if len(gens):
        gen_el = n.generators_t.p[gens].clip(lower=0.0)
        gen_el_country = gen_el.T.groupby(
            n.generators.loc[gens, "bus"].map(bus_country)
        ).sum().T
    else:
        gen_el_country = pd.DataFrame(0.0, index=n.snapshots, columns=countries)

    # Links injecting into AC buses (bus1 is AC)
    links_el = n.links.index[n.links.bus1.isin(ac_buses_c)]
    if len(links_el):
        p1 = _fix_sign(_get_ts(n.links_t, "p1")[links_el]).clip(lower=0.0)
        link_el_country = p1.T.groupby(
            n.links.loc[links_el, "bus1"].map(bus_country)
        ).sum().T
    else:
        link_el_country = pd.DataFrame(0.0, index=n.snapshots, columns=countries)

    E_MWh = (
        gen_el_country
        .add(link_el_country, fill_value=0.0)
        .reindex(columns=countries, fill_value=0.0)
    )
    E_MWh.index = pd.to_datetime(E_MWh.index)

    # -------------------------
    # (B) CO2 flows (tCO2/h) from link extra ports
    # -------------------------
    def co2_to(target_bus_name: str) -> pd.DataFrame:
        co2 = pd.DataFrame(0.0, index=n.snapshots, columns=countries)
        for k in [2, 3, 4]:
            busk, pk = f"bus{k}", f"p{k}"
            if (busk in n.links.columns) and (pk in n.links_t):
                mask = (
                    (n.links[busk] == target_bus_name) &
                    (n.links.bus1.isin(ac_buses_c))
                )
                if mask.any():
                    p = _fix_sign(_get_ts(n.links_t, pk)[mask.index[mask]]).clip(lower=0.0)
                    co2_c = p.T.groupby(
                        n.links.loc[mask.index[mask], "bus1"].map(bus_country)
                    ).sum().T
                    co2 = co2.add(
                        co2_c.reindex(columns=countries, fill_value=0.0), fill_value=0.0
                    )
        co2.index = pd.to_datetime(co2.index)
        return co2

    CO2_atm_t   = co2_to("co2 atmosphere")
    CO2_store_t = co2_to("co2 stored")
    CO2_seq_t   = co2_to("co2 sequestered")

    # -------------------------
    # (C) Carbon intensity (kgCO2/MWh)
    # -------------------------
    if mode == "net":
        co2_numerator = CO2_atm_t
    elif mode == "gross":
        co2_numerator = CO2_atm_t + CO2_store_t + CO2_seq_t
    else:
        raise ValueError("mode must be 'net' or 'gross'")

    # E=0 -> NaN (no division by zero)
    ci = (1000.0 * safe_divide(co2_numerator, E_MWh))

    return {
        "ci":           ci,
        "E_MWh":        E_MWh,
        "CO2_atm_t":    CO2_atm_t,
        "CO2_store_t":  CO2_store_t,
        "CO2_seq_t":    CO2_seq_t,
    }


def export_scenario_ci_csv(
    path_nc: str,
    out_dir: str,
    countries: list = COUNTRIES,
) -> tuple[dict, dict, pd.Series]:
    """
    Load network once, extract CI (net + gross), export two CSV files.

    Returns
    -------
    out_net  : dict from extract_ac_country_carbon_intensity (mode='net')
    out_gro  : dict from extract_ac_country_carbon_intensity (mode='gross')
    snapshot_hours : pd.Series (needed by summarize_ci)
    """
    os.makedirs(out_dir, exist_ok=True)

    n = pypsa.Network(path_nc)
    snapshot_hours = get_snapshot_hours(n)

    out_net = extract_ac_country_carbon_intensity(n, countries=countries, mode="net")
    out_gro = extract_ac_country_carbon_intensity(n, countries=countries, mode="gross")

    scen = scenario_name_from_path(path_nc)

    net_csv = os.path.join(out_dir, f"carbon_intensity_AC_net_{scen}.csv")
    gro_csv = os.path.join(out_dir, f"carbon_intensity_AC_gross_{scen}.csv")

    out_net["ci"].to_csv(net_csv, index=True)
    out_gro["ci"].to_csv(gro_csv, index=True)

    print(f"[OK] {scen:<45}  shape={out_net['ci'].shape}  net -> {os.path.basename(net_csv)}")
    print(f"     {'':<45}                          gross -> {os.path.basename(gro_csv)}")

    return out_net, out_gro, snapshot_hours


def summarize_ci(
    out: dict,
    snapshot_hours: pd.Series,
    countries: list = COUNTRIES,
    thresholds: list = CI_THRESHOLDS,
) -> pd.DataFrame:
    """
    Summary statistics per country for one scenario (net or gross).

    Metrics:
      mean_hourly, max_hourly, p99_hourly
      hours_gt_X  for each threshold (real hours, not snapshot count)
      annual_energy_weighted  (kgCO2/MWh): 1000 * sum(CO2_atm) / sum(E)
      share_E0    fraction of snapshots with zero production
      share_nan_ci fraction of snapshots with NaN intensity
    """
    ci  = out["ci"].reindex(columns=countries)
    E   = out["E_MWh"].reindex(columns=countries)
    CO2 = out["CO2_atm_t"].reindex(columns=countries)

    rows = []
    for c in countries:
        s   = ci[c]
        e   = E[c]
        co2 = CO2[c]

        annual_weighted = np.nan
        if e.sum() > 0:
            annual_weighted = 1000.0 * co2.sum() / e.sum()

        row = {
            "country":                c,
            "mean_hourly":            s.mean(),
            "max_hourly":             s.max(),
            "p99_hourly":             s.quantile(0.99),
            "annual_energy_weighted": annual_weighted,
            "share_E0":               (e == 0).mean(),
            "share_nan_ci":           s.isna().mean(),
        }

        # hours above threshold — uses snapshot_hours for bi-hourly correctness
        for thr in thresholds:
            row[f"hours_gt_{thr}"] = snapshot_hours[s > thr].sum()

        rows.append(row)

    return pd.DataFrame(rows)


# -------------------------------------------------------------------
# MAIN LOOP
# -------------------------------------------------------------------
all_summaries = []

for path_nc in nc_files:
    if not os.path.exists(path_nc):
        print(f"[SKIP] missing file: {path_nc}")
        continue

    out_net, out_gro, snapshot_hours = export_scenario_ci_csv(
        path_nc,
        out_dir,
        countries=COUNTRIES,
    )

    scen = scenario_name_from_path(path_nc)

    summ_net = summarize_ci(out_net, snapshot_hours, countries=COUNTRIES)
    summ_net["scenario"] = scen
    summ_net["mode"] = "net"

    summ_gro = summarize_ci(out_gro, snapshot_hours, countries=COUNTRIES)
    summ_gro["scenario"] = scen
    summ_gro["mode"] = "gross"

    all_summaries.extend([summ_net, summ_gro])

if all_summaries:
    summary_df = pd.concat(all_summaries, ignore_index=True)
    summary_path = os.path.join(out_dir, "summary_carbon_intensity_AC.csv")
    summary_df.to_csv(summary_path, index=False)
    print(f"\n[OK] Saved summary: {summary_path}")

**Current prices from ENTSOE**
Fetches prices and fees, and save to CSV in the same format as the others

In [5]:
import pandas as pd

TSO_TARIFFS = {
    "BE": 2.62,   # €/MWh
    "DE": 5.35,   # €/MWh
    "NL": 0.00,   # €/MWh
}

# ── Caricamento e parsing CSV ENTSO-E ───────────────────────────────────────

def parse_entso_index(index):
    starts = index.str.split(" - ").str[0]
    starts = starts.str.replace(r"\s*\(.*?\)", "", regex=True).str.strip()
    return pd.to_datetime(starts, format="%d/%m/%Y %H:%M:%S")

DE = pd.read_csv(r"data\carriers_data\electricity\DE.csv", index_col=0)
BE = pd.read_csv(r"data\carriers_data\electricity\BE.csv", index_col=0)
NL = pd.read_csv(r"data\carriers_data\electricity\NL.csv", index_col=0)

DE = DE[DE["Sequence"] == "Sequence Sequence 1"][["Day-ahead Price (EUR/MWh)"]]
BE = BE[["Day-ahead Price (EUR/MWh)"]]
NL = NL[["Day-ahead Price (EUR/MWh)"]]

DE.index = parse_entso_index(DE.index)
BE.index = parse_entso_index(BE.index)
NL.index = parse_entso_index(NL.index)

# Resample a orario
DE = DE.resample("2h").mean()
BE = BE.resample("2h").mean()
NL = NL.resample("2h").mean()

# ── Somma fee al prezzo day-ahead ────────────────────────────────────────────
print(DE.head())
print(BE.head())
print(NL.head())

DE["Price + Fees (EUR/MWh)"] = DE["Day-ahead Price (EUR/MWh)"] + TSO_TARIFFS["DE"]
BE["Price + Fees (EUR/MWh)"] = BE["Day-ahead Price (EUR/MWh)"] + TSO_TARIFFS["BE"]
NL["Price + Fees (EUR/MWh)"] = NL["Day-ahead Price (EUR/MWh)"] + TSO_TARIFFS["NL"]

# Create a bihourly index referred to 2013
index = pd.date_range(start="2013-01-01", end="2013-12-31 23:00:00", freq="2h")
final_df = pd.DataFrame(index=index, columns = ['NL', 'BE', 'DE'])
final_df['NL'] = NL["Price + Fees (EUR/MWh)"].values
final_df['BE'] = BE["Price + Fees (EUR/MWh)"].values
final_df['DE'] = DE["Price + Fees (EUR/MWh)"].values

# Save as CSV
final_df.to_csv(r"data\carriers_data\electricity\prices_AC_allin_2025_2h.csv", index=True)

                     Day-ahead Price (EUR/MWh)
MTU (CET/CEST)                                
2025-01-01 00:00:00                      1.880
2025-01-01 02:00:00                     -0.005
2025-01-01 04:00:00                     -0.035
2025-01-01 06:00:00                     -0.050
2025-01-01 08:00:00                     -0.025
                     Day-ahead Price (EUR/MWh)
MTU (CET/CEST)                                
2025-01-01 00:00:00                     10.445
2025-01-01 02:00:00                      7.495
2025-01-01 04:00:00                      2.650
2025-01-01 06:00:00                      3.035
2025-01-01 08:00:00                      5.870
                     Day-ahead Price (EUR/MWh)
MTU (CET/CEST)                                
2025-01-01 00:00:00                      9.930
2025-01-01 02:00:00                      3.720
2025-01-01 04:00:00                      0.340
2025-01-01 06:00:00                      0.775
2025-01-01 08:00:00                      4.695


In [8]:
import pandas as pd

DE = pd.read_csv(r"data\carriers_data\electricity\DE_carb_2024_hourly.csv", index_col=0)
BE = pd.read_csv(r"data\carriers_data\electricity\BE_carb_2024_hourly.csv", index_col=0)
NL = pd.read_csv(r"data\carriers_data\electricity\NL_carb_2024_hourly.csv", index_col=0)

DE = DE['Carbon Intensity gCO₂eq/kWh (LCA)']
BE = BE['Carbon Intensity gCO₂eq/kWh (LCA)']
NL = NL['Carbon Intensity gCO₂eq/kWh (LCA)']

# Convert index to datetime
DE.index = pd.to_datetime(DE.index)
BE.index = pd.to_datetime(BE.index)
NL.index = pd.to_datetime(NL.index)

# Resample to bihourly
DE = DE.resample("2h").mean()
BE = BE.resample("2h").mean()
NL = NL.resample("2h").mean()

# Cut 29th Feb 2024 (leap year)
DE = DE[~((DE.index.month == 2) & (DE.index.day == 29))]
BE = BE[~((BE.index.month == 2) & (BE.index.day == 29))]
NL = NL[~((NL.index.month == 2) & (NL.index.day == 29))]

# Create 2013 bihourly index
index = pd.date_range(start="2013-01-01", end="2013-12-31 23:00:00", freq="2h")
final_df = pd.DataFrame(index=index, columns=['NL', 'BE', 'DE'])

# Assign values, truncating to match target length if needed
final_df['NL'] = NL.values
final_df['BE'] = BE.values
final_df['DE'] = DE.values

# Save to CSV
final_df.to_csv(r"data\carriers_data\electricity\carbon_intensity_AC_net_2025_2h.csv", index=True)


Resample prices csv in 2h

In [2]:
import pandas as pd

# Files
prices_files = [
    r"data\carriers_data\electricity\prices_AC_allin_rigid_2030.csv",
    r"data\carriers_data\electricity\prices_AC_allin_rigid_2040.csv",
    r"data\carriers_data\electricity\prices_AC_allin_rigid_2050.csv",
    r"data\carriers_data\electricity\prices_AC_allin_flexible_2030.csv",
    r"data\carriers_data\electricity\prices_AC_allin_flexible_2040.csv",
    r"data\carriers_data\electricity\prices_AC_allin_flexible_2050.csv",
    r"data\carriers_data\electricity\prices_AC_allin_flexible-moderate_2030.csv",
    r"data\carriers_data\electricity\prices_AC_allin_flexible-moderate_2040.csv",
    r"data\carriers_data\electricity\prices_AC_allin_flexible-moderate_2050.csv",
    r"data\carriers_data\electricity\prices_AC_allin_retro-tes_2030.csv",
    r"data\carriers_data\electricity\prices_AC_allin_retro-tes_2040.csv",
    r"data\carriers_data\electricity\prices_AC_allin_retro-tes_2050.csv",
    r"data\carriers_data\electricity\carbon_intensity_AC_net_rigid_2030.csv",
    r"data\carriers_data\electricity\carbon_intensity_AC_net_rigid_2040.csv",
    r"data\carriers_data\electricity\carbon_intensity_AC_net_rigid_2050.csv",
    r"data\carriers_data\electricity\carbon_intensity_AC_net_flexible_2030.csv",
    r"data\carriers_data\electricity\carbon_intensity_AC_net_flexible_2040.csv",
    r"data\carriers_data\electricity\carbon_intensity_AC_net_flexible_2050.csv",
    r"data\carriers_data\electricity\carbon_intensity_AC_net_flexible-moderate_2030.csv",
    r"data\carriers_data\electricity\carbon_intensity_AC_net_flexible-moderate_2040.csv",
    r"data\carriers_data\electricity\carbon_intensity_AC_net_flexible-moderate_2050.csv",
    r"data\carriers_data\electricity\carbon_intensity_AC_net_retro-tes_2030.csv",
    r"data\carriers_data\electricity\carbon_intensity_AC_net_retro-tes_2040.csv",
    r"data\carriers_data\electricity\carbon_intensity_AC_net_retro-tes_2050.csv",
    r"data\carriers_data\electricity\carbon_intensity_AC_gross_rigid_2030.csv",
    r"data\carriers_data\electricity\carbon_intensity_AC_gross_rigid_2040.csv",
    r"data\carriers_data\electricity\carbon_intensity_AC_gross_rigid_2050.csv",
    r"data\carriers_data\electricity\carbon_intensity_AC_gross_flexible_2030.csv",
    r"data\carriers_data\electricity\carbon_intensity_AC_gross_flexible_2040.csv",
    r"data\carriers_data\electricity\carbon_intensity_AC_gross_flexible_2050.csv",
    r"data\carriers_data\electricity\carbon_intensity_AC_gross_flexible-moderate_2030.csv",
    r"data\carriers_data\electricity\carbon_intensity_AC_gross_flexible-moderate_2040.csv",
    r"data\carriers_data\electricity\carbon_intensity_AC_gross_flexible-moderate_2050.csv",
    r"data\carriers_data\electricity\carbon_intensity_AC_gross_retro-tes_2030.csv",
    r"data\carriers_data\electricity\carbon_intensity_AC_gross_retro-tes_2040.csv",
    r"data\carriers_data\electricity\carbon_intensity_AC_gross_retro-tes_2050.csv"
]

# Load in df
dfs = []
for f in prices_files:
    df = pd.read_csv(f, index_col=0)
    df.index = pd.to_datetime(df.index)
    df = df.resample("2h").mean()
    # save with new name adding _2h before .csv
    new_name = f.replace(".csv", "_2h.csv")
    df.to_csv(new_name, index=True)

